# 01 · Ejemplo resuelto: dos puntos objeto, cuatro rayos

Actividad 3 y entregable E4. Todo se calcula **como con calculadora**, con un método independiente del trazador: la intersección con cada cara sale de una ecuación cuadrática y la refracción, de la ley de Snell escrita con ángulos (ver la [deducción](../docs/deduccion.md), secciones 3 y 9). Al final se compara con el trazador vectorial del paquete.

**Lente.** Biconvexa de caras parabólicas, con $n_L = 1.5$:

$$g(x) = -1 + 0.05\,x^2 \quad\text{(cara de entrada)},\qquad f(x) = 1 - 0.05\,x^2 \quad\text{(cara de salida)}.$$

**Rayos.** Los ángulos $\alpha$ se miden desde el eje $+y$, positivos hacia $+x$, y la dirección es $\hat u = (\sin\alpha, \cos\alpha)$.

| Rayo | Sale de | $\alpha$ |
|---|---|---|
| A | $P_1 = (0, -30)$ | $+2^\circ$ |
| B | $P_1 = (0, -30)$ | $-2^\circ$ |
| C | $P_2 = (-1, -30)$ | $0^\circ$ |
| D | $P_2 = (-1, -30)$ | $+4^\circ$ |

## Lo que se calcula en cada cara

Para una cara $y = y_v + a\,x^2$ y un rayo que sale de $(x_0, y_0)$ con dirección $\hat u = (u_x, u_y)$:

1. **Cuadrática:** $A = a\,u_x^2$, $B = 2a\,x_0u_x - u_y$, $C = a\,x_0^2 + y_v - y_0$ y $\Delta = B^2 - 4AC$.
2. **Raíz estable:** $q = -\tfrac12\left(B + \operatorname{sgn}(B)\sqrt{\Delta}\right)$; $\lambda$ es la menor positiva entre $q/A$ y $C/q$, y el impacto es $Q = (x_0, y_0) + \lambda\,\hat u$.
3. **Inclinación de la cara:** $s'(x_Q) = 2a\,x_Q$ y $\varphi = \arctan s'(x_Q)$.
4. **Snell:** $\theta_i = \alpha + \varphi$, $\ \sin\theta_t = (n_1/n_2)\sin\theta_i$, y la nueva dirección es $\alpha' = \theta_t - \varphi$.

In [1]:
import math

import numpy as np

from lentes import imagen_de_dos_rayos, imagen_lente_delgada, imagen_paraxial, lente_biconvexa, trazar
from lentes.a_mano import cruce_de_dos_rayos, trazar_a_mano

lente = lente_biconvexa(R1=10, R2=10, espesor=2, n=1.5, k1=-1, k2=-1)  # la misma lente, para comparar


def mostrar(nombre, rayo):
    x0, y0 = rayo.P
    print(f"Rayo {nombre}: sale de P = ({x0:g}, {y0:g}) con α = {math.degrees(rayo.alpha0):g}°")
    for c in rayo.caras:
        medios = "aire → vidrio" if c.cara == "g" else "vidrio → aire"
        signo = "+" if c.a > 0 else "-"
        print(f"\n  Cara {c.cara} ({medios}; n1 = {c.n1:g}, n2 = {c.n2:g}):  y = {c.y_v:g} {signo} {abs(c.a):g} x²")
        print(f"    û = ({c.u_in[0]:.6f}, {c.u_in[1]:.6f})")
        print(f"    A = {c.A:.6e}   B = {c.B:.6f}   C = {c.C:.6f}   Δ = {c.discriminante:.6f}")
        print(f"    λ = {c.lam:.6f}   →   Q = ({c.Q[0]:.6f}, {c.Q[1]:.6f})")
        print(f"    s'(x_Q) = {c.pendiente:.6f}   φ = {math.degrees(c.phi):.4f}°")
        print(f"    θi = α + φ = {math.degrees(c.theta_i):.4f}°   sen θt = {c.n1 / c.n2:.6f} · sen θi = "
              f"{math.sin(c.theta_t):.6f}   θt = {math.degrees(c.theta_t):.4f}°")
        print(f"    α' = θt - φ = {math.degrees(c.alpha_out):.4f}°   →   û' = ({c.u_out[0]:.6f}, {c.u_out[1]:.6f})")
    if rayo.cruce_eje is not None:
        print(f"\n  El rayo emergente cruza el eje en y = {rayo.cruce_eje:.6f}")

## Punto objeto $P_1 = (0, -30)$, sobre el eje

El rayo A, paso a paso. El rayo B es su espejo ($x \to -x$): mismos números con los signos de $x$ y de los ángulos cambiados.

In [2]:
A = trazar_a_mano((0, -30), math.radians(+2))
B = trazar_a_mano((0, -30), math.radians(-2))
mostrar("A", A)

Rayo A: sale de P = (0, -30) con α = 2°

  Cara g (aire → vidrio; n1 = 1, n2 = 1.5):  y = -1 + 0.05 x²
    û = (0.034899, 0.999391)
    A = 6.089874e-05   B = -0.999391   C = 29.000000   Δ = 0.991718
    λ = 29.069169   →   Q = (1.014499, -0.948540)
    s'(x_Q) = 0.101450   φ = 5.7928°
    θi = α + φ = 7.7928°   sen θt = 0.666667 · sen θi = 0.090394   θt = 5.1863°
    α' = θt - φ = -0.6065°   →   û' = (-0.010586, 0.999944)

  Cara f (vidrio → aire; n1 = 1.5, n2 = 1):  y = 1 - 0.05 x²
    û = (-0.010586, 0.999944)
    A = -5.602990e-06   B = -0.998870   C = 1.897079   Δ = 0.997784
    λ = 1.899205   →   Q = (0.994395, 0.950559)
    s'(x_Q) = -0.099439   φ = -5.6788°
    θi = α + φ = -6.2853°   sen θt = 1.500000 · sen θi = -0.164220   θt = -9.4519°
    α' = θt - φ = -3.7731°   →   û' = (-0.065806, 0.997832)

  El rayo emergente cruza el eje en y = 16.028886


In [3]:
mostrar("B", B)

Rayo B: sale de P = (0, -30) con α = -2°

  Cara g (aire → vidrio; n1 = 1, n2 = 1.5):  y = -1 + 0.05 x²
    û = (-0.034899, 0.999391)
    A = 6.089874e-05   B = -0.999391   C = 29.000000   Δ = 0.991718
    λ = 29.069169   →   Q = (-1.014499, -0.948540)
    s'(x_Q) = -0.101450   φ = -5.7928°
    θi = α + φ = -7.7928°   sen θt = 0.666667 · sen θi = -0.090394   θt = -5.1863°
    α' = θt - φ = 0.6065°   →   û' = (0.010586, 0.999944)

  Cara f (vidrio → aire; n1 = 1.5, n2 = 1):  y = 1 - 0.05 x²
    û = (0.010586, 0.999944)
    A = -5.602990e-06   B = -0.998870   C = 1.897079   Δ = 0.997784
    λ = 1.899205   →   Q = (-0.994395, 0.950559)
    s'(x_Q) = 0.099439   φ = 5.6788°
    θi = α + φ = 6.2853°   sen θt = 1.500000 · sen θi = 0.164220   θt = 9.4519°
    α' = θt - φ = 3.7731°   →   û' = (0.065806, 0.997832)

  El rayo emergente cruza el eje en y = 16.028886


### Imagen de $P_1$

Los rayos emergentes A y B se cortan donde $s\,\hat e_A - t\,\hat e_B = Q_B - Q_A$, que se resuelve con la regla de Cramer (deducción, sección 8). Con $s, t > 0$ el cruce está después de la lente: la imagen es real.

In [4]:
(xi, yi), s, t = cruce_de_dos_rayos(A, B)
par1 = imagen_paraxial(lente, (0, -30))
delg1 = imagen_lente_delgada(lente, (0, -30))
print(f"Imagen de P1 (cruce de A y B):  ({xi + 0.0:.6f}, {yi:.6f})   s = {s:.4f}, t = {t:.4f}")
print(f"Paraxial, lente gruesa:         ({par1.x + 0.0:.6f}, {par1.y:.6f})")
print(f"Paraxial, lente delgada:        y = {delg1.y:.6f}")
print(f"\nEl trazado difiere de la paraxial gruesa en Δy = {yi - par1.y:+.4f}: los rayos entran a ~1 unidad del eje.")
print(f"La lente delgada difiere en {delg1.y - par1.y:+.4f} solo por ignorar el espesor.")

Imagen de P1 (cruce de A y B):  (0.000000, 16.028886)   s = 15.1111, t = 15.1111
Paraxial, lente gruesa:         (0.000000, 16.187166)
Paraxial, lente delgada:        y = 15.000000

El trazado difiere de la paraxial gruesa en Δy = -0.1583: los rayos entran a ~1 unidad del eje.
La lente delgada difiere en -1.1872 solo por ignorar el espesor.


## Punto objeto $P_2 = (-1, -30)$, fuera del eje

El rayo C sale vertical ($\alpha = 0$): en la primera cara $A = 0$ y la cuadrática se vuelve lineal; la fórmula estable lo cubre sin caso especial. El rayo D sale a $4^\circ$. Los dos pasan a lados opuestos del centro de la lente.

In [5]:
C = trazar_a_mano((-1, -30), math.radians(0))
D = trazar_a_mano((-1, -30), math.radians(4))
mostrar("C", C)

Rayo C: sale de P = (-1, -30) con α = 0°

  Cara g (aire → vidrio; n1 = 1, n2 = 1.5):  y = -1 + 0.05 x²
    û = (0.000000, 1.000000)
    A = 0.000000e+00   B = -1.000000   C = 29.050000   Δ = 1.000000
    λ = 29.050000   →   Q = (-1.000000, -0.950000)
    s'(x_Q) = -0.100000   φ = -5.7106°
    θi = α + φ = -5.7106°   sen θt = 0.666667 · sen θi = -0.066336   θt = -3.8036°
    α' = θt - φ = 1.9070°   →   û' = (0.033278, 0.999446)

  Cara f (vidrio → aire; n1 = 1.5, n2 = 1):  y = 1 - 0.05 x²
    û = (0.033278, 0.999446)
    A = -5.537109e-05   B = -0.996118   C = 1.900000   Δ = 0.992673
    λ = 1.907202   →   Q = (-0.936532, 0.956145)
    s'(x_Q) = 0.093653   φ = 5.3503°
    θi = α + φ = 7.2574°   sen θt = 1.500000 · sen θi = 0.189490   θt = 10.9230°
    α' = θt - φ = 5.5727°   →   û' = (0.097108, 0.995274)

  El rayo emergente cruza el eje en y = 10.554764


In [6]:
mostrar("D", D)

Rayo D: sale de P = (-1, -30) con α = 4°

  Cara g (aire → vidrio; n1 = 1, n2 = 1.5):  y = -1 + 0.05 x²
    û = (0.069756, 0.997564)
    A = 2.432983e-04   B = -1.004540   C = 29.050000   Δ = 0.980829
    λ = 29.124155   →   Q = (1.031598, -0.946790)
    s'(x_Q) = 0.103160   φ = 5.8898°
    θi = α + φ = 9.8898°   sen θt = 0.666667 · sen θi = 0.114502   θt = 6.5749°
    α' = θt - φ = 0.6851°   →   û' = (0.011958, 0.999929)

  Cara f (vidrio → aire; n1 = 1.5, n2 = 1):  y = 1 - 0.05 x²
    û = (0.011958, 0.999929)
    A = -7.149152e-06   B = -1.001162   C = 1.893580   Δ = 1.002380
    λ = 1.891357   →   Q = (1.054214, 0.944432)
    s'(x_Q) = -0.105421   φ = -6.0180°
    θi = α + φ = -5.3328°   sen θt = 1.500000 · sen θi = -0.139412   θt = -8.0138°
    α' = θt - φ = -1.9958°   →   û' = (-0.034827, 0.999393)

  El rayo emergente cruza el eje en y = 31.196066


### Imagen de $P_2$

In [7]:
(xi2, yi2), s2, t2 = cruce_de_dos_rayos(C, D)
par2 = imagen_paraxial(lente, (-1, -30))
print(f"Imagen de P2 (cruce de C y D):  ({xi2:.6f}, {yi2:.6f})   s = {s2:.4f}, t = {t2:.4f}")
print(f"Paraxial, lente gruesa:         ({par2.x:.6f}, {par2.y:.6f})   aumento lateral = {par2.aumento:.6f}")
print(f"\nLa imagen queda al otro lado del eje (invertida) y unas {abs(xi2):.2f} veces la altura del objeto.")

Imagen de P2 (cruce de C y D):  (0.530011, 15.986908)   s = 15.1021, t = 15.0516
Paraxial, lente gruesa:         (0.534759, 16.187166)   aumento lateral = -0.534759

La imagen queda al otro lado del eje (invertida) y unas 0.53 veces la altura del objeto.


## Comprobación con el trazador vectorial

El paquete traza los mismos cuatro rayos por otro camino: intersección numérica (método de Brent) y ley de Snell vectorial con la normal orientada hacia el medio de incidencia. Si las dos implementaciones coinciden, el ejemplo resuelto y la simulación se validan mutuamente (también está como prueba automática en `tests/test_a_mano.py`).

In [8]:
maxima = 0.0
trazados = {}
for nombre, rayo in (("A", A), ("B", B), ("C", C), ("D", D)):
    tr = trazar(lente, rayo.P, [math.sin(rayo.alpha0), math.cos(rayo.alpha0)])
    trazados[nombre] = tr
    for paso, Q, d in zip(rayo.caras, tr.puntos[1:], tr.direcciones[1:]):
        maxima = max(maxima, np.abs(np.subtract(paso.Q, Q)).max(), np.abs(np.subtract(paso.u_out, d)).max())

img1 = imagen_de_dos_rayos(trazados["A"], trazados["B"])
img2 = imagen_de_dos_rayos(trazados["C"], trazados["D"])
print(f"Máxima diferencia entre los dos métodos (puntos y direcciones): {maxima:.1e}")
print(f"Imagen de P1 con el trazador: ({img1.punto[0] + 0.0:.6f}, {img1.punto[1]:.6f})")
print(f"Imagen de P2 con el trazador: ({img2.punto[0]:.6f}, {img2.punto[1]:.6f})")

Máxima diferencia entre los dos métodos (puntos y direcciones): 2.2e-16
Imagen de P1 con el trazador: (0.000000, 16.028886)
Imagen de P2 con el trazador: (0.530011, 15.986908)


## Qué muestra el ejemplo

- Cada rayo se refracta dos veces: al entrar se acerca a la normal ($\theta_t < \theta_i$) y al salir se aleja ($\theta_t > \theta_i$).
- La imagen de $P_1$ queda un poco antes de la predicción paraxial de lente gruesa, porque los rayos entran a una unidad del eje: es la aberración esférica que se estudia en la actividad 5. La ecuación de lente delgada se equivoca bastante más, solo por ignorar el espesor.
- La imagen de $P_2$ sale invertida y reducida, como predice la paraxial; la pequeña diferencia viene de la apertura de los rayos y de la altura del objeto.
- Los dos métodos coinciden hasta la precisión de la máquina.

Para el póster bastan las tablas de los rayos A y C y las dos imágenes.